## Install Dependencies

In [17]:
# Install (Colab). Skip if already installed.
!pip install -q transformers accelerate sentencepiece rouge-score pandas
# Only needed for the backend you actually use:
# !pip install -q google-genai        # current Gemini SDK
# !pip install -q google-generativeai # legacy Gemini SDK
# !pip install -q openai              # Azure OpenAI


## Import Libraries

In [18]:
import os
import re
import json
import time
import platform
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
from rouge_score import rouge_scorer
from IPython.display import display, Markdown

import warnings
warnings.filterwarnings("ignore")

# torch / transformers are imported lazily inside the huggingface backend so
# that the mock and API backends run without them.
print("Imports OK")


Imports OK


## Configuration

In [19]:
BACKEND = "huggingface"          # 'mock' | 'huggingface' | 'google' | 'azure'
N_REPEATS = 1             # repeats per (problem, method); >1 recommended for API backends

HF_MODEL = "google/flan-t5-base"   # try 'google/flan-t5-large' as a middle data point

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.0-flash")

AZURE_API_KEY = os.environ.get("AZURE_API_KEY", "")
AZURE_ENDPOINT = os.environ.get("AZURE_ENDPOINT", "")
AZURE_API_VERSION = os.environ.get("AZURE_API_VERSION", "2024-06-01")
AZURE_DEPLOYMENT = os.environ.get("AZURE_DEPLOYMENT", "")

if BACKEND not in ("mock", "huggingface", "google", "azure"):
    raise ValueError("BACKEND must be 'mock', 'huggingface', 'google' or 'azure'")

print(f"Backend: {BACKEND} | repeats: {N_REPEATS}")


Backend: huggingface | repeats: 1


## LLMWrapper Class Definition

In [16]:
class LLMWrapper:
    """Unified interface over a mock, HuggingFace seq2seq, Gemini, and Azure OpenAI."""

    def __init__(self, backend: str, model: Optional[str] = None):
        self.backend = backend
        self.model_name = model
        self.call_count = 0
        self.max_input_tokens: Optional[int] = None
        self._setup()

    # ---------- setup ----------

    def _setup(self):
        if self.backend == "mock":
            self.model_name = self.model_name or "mock-v1"
            self.max_input_tokens = 512          # exercises the fitting path
            print("Mock backend ready. No model loaded.")

        elif self.backend == "huggingface":
            import torch
            from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
            self._torch = torch
            if not self.model_name:
                raise ValueError("model name required for huggingface backend")
            print(f"Loading Hugging Face model: {self.model_name} ...")
            self._tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self._hf_model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)
            self._hf_model.eval()
            self._device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            self._hf_model.to(self._device)
            cap = getattr(self._tokenizer, "model_max_length", 512)
            # some tokenizers report a sentinel like 1e30; fall back to 512
            self.max_input_tokens = int(cap) if isinstance(cap, int) and 0 < cap < 100_000 else 512
            print(f"Loaded on {self._device}. Input cap: {self.max_input_tokens} tokens.")

        elif self.backend == "google":
            if not GOOGLE_API_KEY:
                raise ValueError("GOOGLE_API_KEY is not set.")
            try:
                from google import genai as genai_new
                self._client = genai_new.Client(api_key=GOOGLE_API_KEY)
                self._flavor = "google-genai"
            except ImportError:
                import google.generativeai as genai_legacy
                genai_legacy.configure(api_key=GOOGLE_API_KEY)
                self._genai = genai_legacy
                self._gemini = genai_legacy.GenerativeModel(self.model_name)
                self._flavor = "google-generativeai"
            print(f"Gemini client ready ({self._flavor}). Model: {self.model_name}")

        elif self.backend == "azure":
            from openai import AzureOpenAI
            missing = [n for n, v in [
                ("AZURE_API_KEY", AZURE_API_KEY),
                ("AZURE_ENDPOINT", AZURE_ENDPOINT),
                ("AZURE_DEPLOYMENT", AZURE_DEPLOYMENT),
            ] if not v]
            if missing:
                raise ValueError(f"Missing Azure config: {', '.join(missing)}")
            self._client = AzureOpenAI(
                azure_endpoint=AZURE_ENDPOINT,
                api_key=AZURE_API_KEY,
                api_version=AZURE_API_VERSION,
            )
            self.model_name = AZURE_DEPLOYMENT
            print(f"Azure OpenAI client ready. Deployment: {self.model_name}")

    def describe(self) -> str:
        return f"{self.backend}:{self.model_name}"

    # ---------- token counting ----------

    def count_tokens(self, text: str) -> int:
        """Real count for huggingface, word count for mock, rough proxy otherwise."""
        if self.backend == "huggingface":
            return len(self._tokenizer(text, truncation=False)["input_ids"])
        return len(text.split())

    # ---------- generation ----------

    def generate(self, prompt: str, max_tokens: int = 700, temperature: float = 0.0) -> str:
        self.call_count += 1

        if self.backend == "mock":
            return MOCK_RESPONSE

        if self.backend == "huggingface":
            enc = self._tokenizer(prompt, return_tensors="pt",
                                  truncation=True, max_length=self.max_input_tokens)
            enc = {k: v.to(self._device) for k, v in enc.items()}
            kwargs: Dict[str, Any] = {"max_new_tokens": max_tokens}
            if temperature and temperature > 0:
                kwargs.update(do_sample=True, temperature=float(temperature))
            else:
                kwargs.update(do_sample=False)
            with self._torch.inference_mode():
                out = self._hf_model.generate(**enc, **kwargs)
            return self._tokenizer.decode(out[0], skip_special_tokens=True).strip()

        if self.backend == "google":
            if self._flavor == "google-genai":
                resp = self._client.models.generate_content(
                    model=self.model_name,
                    contents=prompt,
                    config={"temperature": temperature, "max_output_tokens": max_tokens},
                )
                return (getattr(resp, "text", "") or "").strip()
            cfg = self._genai.types.GenerationConfig(
                max_output_tokens=max_tokens, temperature=temperature
            )
            resp = self._gemini.generate_content(prompt, generation_config=cfg)
            if not getattr(resp, "candidates", None):
                return ""
            parts = getattr(resp.candidates[0].content, "parts", []) or []
            return "".join(getattr(p, "text", "") or "" for p in parts).strip()

        if self.backend == "azure":
            base = dict(model=self.model_name,
                        messages=[{"role": "user", "content": prompt}])
            try:
                resp = self._client.chat.completions.create(
                    **base, max_tokens=max_tokens, temperature=temperature)
            except Exception:
                # Newer reasoning deployments reject max_tokens and temperature.
                resp = self._client.chat.completions.create(
                    **base, max_completion_tokens=max_tokens)
            content = resp.choices[0].message.content
            return content.strip() if content else ""

        raise ValueError(f"Unknown backend: {self.backend}")


# Canned mock answer: correct for P1 only, so the grader is exercised on both
# a passing and a failing case.
MOCK_RESPONSE = """Q1 net profit 4800. Q2 product B at 39.3%. Q3 product C, 12 units.

{"q1_value": 4800, "q2_label": "B", "q2_value": 39.3, "q3_label": "C", "q3_value": 12}"""

active_model = {"mock": "mock-v1", "huggingface": HF_MODEL,
                "google": GEMINI_MODEL, "azure": AZURE_DEPLOYMENT}[BACKEND]
llm = LLMWrapper(backend=BACKEND, model=active_model)


Loading Hugging Face model: google/flan-t5-base ...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded on cpu. Input cap: 512 tokens.


## Run Metadata

In [20]:
def _ver(mod: str) -> str:
    try:
        import importlib.metadata as md
        return md.version(mod)
    except Exception:
        return "not installed"

RUN_META = {
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "python": platform.python_version(),
    "backend": BACKEND,
    "model": llm.describe(),
    "max_input_tokens": llm.max_input_tokens,
    "n_repeats": N_REPEATS,
    "transformers": _ver("transformers"),
    "torch": _ver("torch"),
    "rouge_score": _ver("rouge-score"),
    "pandas": _ver("pandas"),
}
for k, v in RUN_META.items():
    print(f"{k:20s} {v}")


timestamp_utc        2026-08-12T10:03:58Z
python               3.12.13
backend              huggingface
model                huggingface:google/flan-t5-base
max_input_tokens     512
n_repeats            1
transformers         5.13.1
torch                2.11.0+cpu
rouge_score          0.1.2
pandas               2.2.2


## Evaluation Problems

In [21]:
ANSWER_KEYS = ["q1_value", "q2_label", "q2_value", "q3_label", "q3_value"]

EVAL_SET: List[Dict[str, Any]] = [
    {
        "id": "P1_product_mix",
        "text": """A startup sells 3 products.
- Product A: costs $120 to make, sells for $180
- Product B: costs $85 to make, sells for $140
- Product C: costs $200 to make, sells for $280

Last month they sold 50 units of A, 80 units of B, and 30 units of C.
Fixed monthly costs are $5,000.

Q1. What was the net profit last month, after fixed costs?
Q2. Which single product has the highest profit margin as a percentage of its
    selling price, and what is that percentage?
Q3. To raise next month's net profit by at least 20% (fixed costs unchanged) by
    increasing sales of only ONE product, which product requires the FEWEST
    additional units, and how many additional units? Round up to whole units.""",
        "ground_truth": {
            "q1_value": 4800, "q2_label": "B", "q2_value": 39.3,
            "q3_label": "C", "q3_value": 12,
        },
        "label_vocab": ["A", "B", "C"],
        "reference_text": (
            "Per-unit profit: A 60, B 55, C 80. Monthly: A 3000, B 4400, C 2400. "
            "Gross profit 9800, minus 5000 fixed costs, net profit 4800. "
            "Margins: A 33.3%, B 39.3%, C 28.6%, so B is highest. "
            "Target profit 5760, so 960 additional profit needed: 16 units of A, "
            "18 of B, or 12 of C. Product C needs the fewest additional units, 12."
        ),
    },
    {
        "id": "P2_academy_costs",
        "text": """An AI training academy runs 2 programs.
- Data Science: 40 students, each using $15 of cloud compute credits per month.
- GenAI: 60 students, using an LLM API costing $0.02 per 1,000 input tokens and
  $0.06 per 1,000 output tokens. Each GenAI student processes 500,000 input
  tokens and 100,000 output tokens per month.
- Fixed platform hosting fee: $1,200 per month.

Q1. What is the total monthly cost for both programs combined, including the
    fixed fee?
Q2. Which program has the higher variable cost PER STUDENT, and by how many
    dollars per student?
Q3. If a semantic cache cuts GenAI token costs by 20%, which program then has
    the higher variable cost per student, and what is the new total monthly cost?""",
        "ground_truth": {
            "q1_value": 2760, "q2_label": "GenAI", "q2_value": 1.0,
            "q3_label": "Data Science", "q3_value": 2568,
        },
        "label_vocab": ["Data Science", "GenAI"],
        "reference_text": (
            "Data Science variable cost 40 times 15 equals 600. GenAI per student: "
            "500 times 0.02 equals 10 for input plus 100 times 0.06 equals 6 for "
            "output, so 16 per student, and 60 times 16 equals 960. Total with the "
            "1200 fixed fee is 2760. GenAI costs 16 per student versus 15, higher by 1. "
            "After a 20% cut GenAI falls to 12.80 per student, so Data Science is now "
            "higher, and the new total is 600 plus 768 plus 1200, which is 2568."
        ),
    },
    {
        "id": "P3_cafe_sizes",
        "text": """A cafe sells 3 drink sizes.
- Small: costs $8, sells for $20
- Medium: costs $11, sells for $28
- Large: costs $15, sells for $40

Daily sales: 120 small, 90 medium, 60 large. Fixed daily costs: $900.

Q1. What is the daily net profit after fixed costs?
Q2. Which size has the highest profit margin as a percentage of its selling
    price, and what is that percentage?
Q3. To raise daily net profit by at least 10% by increasing sales of only ONE
    size, which size requires the FEWEST additional units, and how many
    additional units? Round up to whole units.""",
        "ground_truth": {
            "q1_value": 3570, "q2_label": "Large", "q2_value": 62.5,
            "q3_label": "Large", "q3_value": 15,
        },
        "label_vocab": ["Small", "Medium", "Large"],
        "reference_text": (
            "Per-unit profit: small 12, medium 17, large 25. Daily: 1440, 1530, 1500. "
            "Gross 4470 minus 900 fixed equals net profit 3570. Margins: small 60%, "
            "medium 60.7%, large 62.5%, so large is highest. A 10% rise needs 357 more "
            "profit: 30 small, 21 medium, or 15 large. Large needs the fewest, 15 units."
        ),
    },
    {
        "id": "P4_saas_tiers",
        "text": """A SaaS product has 3 tiers, with no variable cost per subscriber.
- Basic: $20/month, 500 subscribers, 6% monthly churn
- Pro: $50/month, 200 subscribers, 4% monthly churn
- Enterprise: $200/month, 30 subscribers, 2% monthly churn

Fixed monthly costs: $12,000.

Q1. What is the monthly net profit after fixed costs?
Q2. Which tier loses the most monthly revenue to churn, and how many dollars
    per month does it lose? Treat churned subscribers as an exact fraction, not
    a whole number.
Q3. To add exactly $2,000 to monthly net profit by acquiring new subscribers in
    only ONE tier, which tier requires the FEWEST new subscribers, and how many?""",
        "ground_truth": {
            "q1_value": 14000, "q2_label": "Basic", "q2_value": 600,
            "q3_label": "Enterprise", "q3_value": 10,
        },
        "label_vocab": ["Basic", "Pro", "Enterprise"],
        "reference_text": (
            "Revenue: Basic 10000, Pro 10000, Enterprise 6000, total 26000. "
            "Minus 12000 fixed costs gives net profit 14000. Churned revenue: Basic "
            "30 subscribers times 20 equals 600, Pro 8 times 50 equals 400, Enterprise "
            "0.6 times 200 equals 120, so Basic loses the most at 600. For 2000 more "
            "profit: 100 Basic, 40 Pro, or 10 Enterprise subscribers. Enterprise needs "
            "the fewest, 10."
        ),
    },
]

print(f"{len(EVAL_SET)} problems loaded.")


4 problems loaded.


## Output Contract

In [22]:
OUTPUT_CONTRACT = """
End your response with a single JSON object and nothing after it, in exactly this form:

{
  "q1_value": <number>,
  "q2_label": "<short name>",
  "q2_value": <number>,
  "q3_label": "<short name>",
  "q3_value": <number>
}

Each *_label is the answer to the "which one" part of that question, and each
*_value is the answer to the "how much / how many" part of the same question.

Rules: numbers only, no currency symbols, no thousands separators, no units,
no percent signs. Labels must be the short name only (for example "B", "Large",
"GenAI"), not a sentence.
"""


## Prompt Builders

In [23]:
def fit_prompt(llm: LLMWrapper, head: str, middle: str, tail: str) -> Tuple[str, Dict[str, Any]]:
    """Assemble head+middle+tail, trimming ONLY middle to fit the input cap."""
    limit = llm.max_input_tokens
    full = head + middle + tail
    meta = {"prompt_tokens": llm.count_tokens(full), "input_cap": limit,
            "middle_trimmed": False, "middle_kept_frac": 1.0, "hard_overflow": False}

    if limit is None or meta["prompt_tokens"] <= limit:
        return full, meta

    fixed = llm.count_tokens(head + tail)
    budget = limit - fixed - 8          # small safety margin
    if budget <= 0:
        out = head + tail
        meta.update(middle_trimmed=True, middle_kept_frac=0.0, hard_overflow=True,
                    prompt_tokens=llm.count_tokens(out))
        return out, meta

    words = middle.split()
    lo, hi, best = 0, len(words), 0
    while lo <= hi:                      # binary search on how many words fit
        mid = (lo + hi) // 2
        if llm.count_tokens(" ".join(words[:mid])) <= budget:
            best, lo = mid, mid + 1
        else:
            hi = mid - 1

    out = head + " ".join(words[:best]) + tail
    meta.update(middle_trimmed=True,
                middle_kept_frac=round(best / max(len(words), 1), 3),
                prompt_tokens=llm.count_tokens(out))
    return out, meta


def decomposition_parts(problem: str) -> Tuple[str, str, str]:
    head = ("You are an expert analytical problem solver.\n\n"
            f"PROBLEM:\n{problem}\n\n")
    tail = ("Break the problem above into a numbered list of sequential sub-tasks.\n"
            "Each sub-task must be specific and independently solvable.\n"
            "Do NOT solve anything yet.\n\nSUB-TASKS:")
    return head, "", tail


def solver_parts(problem: str, subtasks: str) -> Tuple[str, str, str]:
    head = ("You are an expert quantitative reasoning assistant.\n\n"
            f"PROBLEM:\n{problem}\n\nSUB-TASKS:\n")
    tail = ("\n\nSolve each sub-task above step by step. Show every calculation.\n"
            "Label each solution with its sub-task number. Be exact with numbers.\n\n"
            "STEP-BY-STEP SOLUTIONS:")
    return head, subtasks, tail


def synthesis_parts(problem: str, solutions: str) -> Tuple[str, str, str]:
    head = ("You are a senior business analyst writing a final answer.\n\n"
            f"PROBLEM:\n{problem}\n\nINTERMEDIATE WORKINGS:\n")
    tail = ("\n\nAnswer Q1, Q2 and Q3 in order, briefly.\n"
            f"{OUTPUT_CONTRACT}\n\nFINAL ANSWER:")
    return head, solutions, tail


def single_shot_parts(problem: str) -> Tuple[str, str, str]:
    head = ("Solve the following problem completely and answer all three questions.\n\n"
            f"{problem}\n\n")
    tail = f"{OUTPUT_CONTRACT}\n\nANSWER:"
    return head, "", tail


print("Prompt builders defined.")


Prompt builders defined.


## Grader and Helper Functions

In [24]:
PLACEHOLDER = re.compile(r"^\s*<.*>\s*$")


def parse_answer(text: str) -> Tuple[Optional[Dict[str, Any]], str]:
    """Return (answer_dict, parse_mode). parse_mode in {'json','regex','fail'}."""
    if not text:
        return None, "fail"

    # Strategy 1: last valid JSON object carrying at least one answer key
    for block in reversed(re.findall(r"\{[^{}]*\}", text, re.DOTALL)):
        try:
            obj = json.loads(block)
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict) and any(k in obj for k in ANSWER_KEYS):
            clean = {k: v for k, v in obj.items()
                     if not (isinstance(v, str) and PLACEHOLDER.match(v))}
            if any(k in clean for k in ANSWER_KEYS):
                return clean, "json"

    # Strategy 2: per-key regex, for models that ignore the contract
    obj = {}
    for key in ANSWER_KEYS:
        m = re.search(rf'"?{key}"?\s*[:=]\s*"?([^",\n}}]+)"?', text)
        if not m:
            continue
        val = m.group(1).strip()
        if PLACEHOLDER.match(val) or "<" in val:   # echoed contract template
            continue
        obj[key] = val
    return (obj, "regex") if obj else (None, "fail")


def _to_float(v: Any) -> Optional[float]:
    if isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if not isinstance(v, str):
        return None
    cleaned = re.sub(r"[^\d.\-]", "", v.replace(",", ""))
    try:
        return float(cleaned)
    except ValueError:
        return None


def _norm_label(v: Any) -> str:
    s = str(v).lower().strip()
    s = re.sub(r"^(product|tier|size|program|the)\s+", "", s)
    return re.sub(r"[^a-z0-9 ]", "", s).strip()


def match_label(predicted: Any, expected: Any, vocab: List[str]) -> bool:
    p, e = _norm_label(predicted), _norm_label(expected)
    if not p:
        return False
    if p == e:
        return True
    hits = [v for v in vocab if re.search(rf"\b{re.escape(_norm_label(v))}\b", p)]
    return len(hits) == 1 and _norm_label(hits[0]) == e   # ambiguous -> wrong


def grade_field(key: str, predicted: Any, expected: Any, vocab: List[str]) -> bool:
    if predicted is None:
        return False
    if key.endswith("_label"):
        return match_label(predicted, expected, vocab)
    p = _to_float(predicted)
    if p is None:
        return False
    e = float(expected)
    tol = max(abs(e) * 0.005, 0.05)      # 0.5% relative, small absolute floor
    return abs(p - e) <= tol


def grade(parsed: Optional[Dict[str, Any]], parse_mode: str,
          ground_truth: Dict[str, Any], vocab: List[str]) -> Dict[str, Any]:
    if parsed is None:
        return {"parse_ok": False, "parse_mode": parse_mode,
                "fields_found": 0, "fields_correct": 0,
                "field_accuracy": 0.0, "fully_correct": False,
                "per_field": {k: False for k in ANSWER_KEYS}}
    per_field = {k: grade_field(k, parsed.get(k), ground_truth[k], vocab)
                 for k in ANSWER_KEYS}
    correct = sum(per_field.values())
    return {"parse_ok": True, "parse_mode": parse_mode,
            "fields_found": sum(1 for k in ANSWER_KEYS if k in parsed),
            "fields_correct": correct,
            "field_accuracy": correct / len(ANSWER_KEYS),
            "fully_correct": correct == len(ANSWER_KEYS),
            "per_field": per_field}


_rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)


def rouge_scores(prediction: str, reference: str) -> Dict[str, float]:
    s = _rouge.score(reference, prediction or "")
    return {"rouge1": round(s["rouge1"].fmeasure, 4),
            "rouge2": round(s["rouge2"].fmeasure, 4),
            "rougeL": round(s["rougeL"].fmeasure, 4)}


def echo_ratio(output: str, problem: str) -> float:
    """Share of output words that also appear in the problem. High = copying."""
    out = [w for w in re.findall(r"[a-z0-9$%.]+", (output or "").lower()) if len(w) > 2]
    if not out:
        return 0.0
    src = set(re.findall(r"[a-z0-9$%.]+", problem.lower()))
    return round(sum(w in src for w in out) / len(out), 4)


print("Grader defined.")


Grader defined.


## Self-test for Grader Functions

In [25]:
def _selftest():
    gt = EVAL_SET[0]["ground_truth"]
    vocab = EVAL_SET[0]["label_vocab"]

    # 1. Echoed contract template must NOT parse as an answer.
    echoed = 'End with: { "q1_value": <number>, "q2_label": "<short name>" }'
    parsed, mode = parse_answer(echoed)
    assert parsed is None and mode == "fail", f"echo leaked through: {parsed}"

    # 2. A real answer parses as json and grades correct.
    good = 'Working... {"q1_value": 4800, "q2_label": "B", "q2_value": 39.3, ' \
           '"q3_label": "C", "q3_value": 12}'
    parsed, mode = parse_answer(good)
    assert mode == "json"
    assert grade(parsed, mode, gt, vocab)["fully_correct"]

    # 3. Formatted numbers still grade correct.
    messy = '{"q1_value": "$4,800", "q2_label": "Product B", "q2_value": "39.29%", ' \
            '"q3_label": "C", "q3_value": "12 units"}'
    parsed, mode = parse_answer(messy)
    assert grade(parsed, mode, gt, vocab)["fully_correct"], "tolerant grading broke"

    # 4. Ambiguous label must FAIL (v2 scored this correct).
    assert not match_label("B, not C", "B", vocab), "ambiguous label passed"
    assert not match_label("GenAI, not Data Science", "Data Science",
                           EVAL_SET[1]["label_vocab"]), "ambiguous label passed"

    # 5. Prompt fitting protects head and tail.
    head, middle, tail = synthesis_parts(EVAL_SET[0]["text"], "filler " * 900)
    prompt, meta = fit_prompt(llm, head, middle, tail)
    if llm.max_input_tokens:
        assert meta["middle_trimmed"] and not meta["hard_overflow"]
        assert "PROBLEM:" in prompt and "FINAL ANSWER:" in prompt, "protected text lost"
        assert meta["prompt_tokens"] <= llm.max_input_tokens

    # 6. Echo detector separates copying from answering.
    assert echo_ratio(EVAL_SET[0]["text"], EVAL_SET[0]["text"]) > 0.9
    assert echo_ratio("zebra xylophone quantum", EVAL_SET[0]["text"]) == 0.0

    print("Self-test passed: parser, grader, prompt fitting, echo detector.")


_selftest()


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2204 > 512). Running this sequence through the model will result in indexing errors


Self-test passed: parser, grader, prompt fitting, echo detector.


## Reasoning Chains and Single Shot Run

In [26]:
class MultiStepReasoningChain:
    """Decompose -> solve -> synthesize, with per-step prompt accounting."""

    def __init__(self, llm: LLMWrapper, verbose: bool = False):
        self.llm = llm
        self.verbose = verbose

    def _step(self, parts, max_tokens: int) -> Tuple[str, Dict[str, Any]]:
        prompt, meta = fit_prompt(self.llm, *parts)
        t0 = time.time()
        out = self.llm.generate(prompt, max_tokens=max_tokens, temperature=0.0)
        meta["gen_seconds"] = round(time.time() - t0, 2)
        meta["out_words"] = len(out.split())
        return out, meta

    def run(self, problem: str) -> Dict[str, Any]:
        t0 = time.time()

        subtasks, m1 = self._step(decomposition_parts(problem), 400)
        solutions, m2 = self._step(solver_parts(problem, subtasks), 700)
        final, m3 = self._step(synthesis_parts(problem, solutions), 500)

        steps = {"decompose": m1, "solve": m2, "synthesize": m3}
        return {
            "subtasks": subtasks,
            "solutions": solutions,
            "final_answer": final,
            "llm_calls": 3,
            "seconds": round(time.time() - t0, 2),
            "steps": steps,
            "any_trim": any(m["middle_trimmed"] for m in steps.values()),
            "any_overflow": any(m["hard_overflow"] for m in steps.values()),
            "max_prompt_tokens": max(m["prompt_tokens"] for m in steps.values()),
            "out_words": m3["out_words"],
        }


def run_single_shot(llm: LLMWrapper, problem: str) -> Dict[str, Any]:
    prompt, meta = fit_prompt(llm, *single_shot_parts(problem))
    t0 = time.time()
    out = llm.generate(prompt, max_tokens=700, temperature=0.0)
    secs = round(time.time() - t0, 2)
    return {"final_answer": out, "llm_calls": 1, "seconds": secs,
            "steps": {"single": meta},
            "any_trim": meta["middle_trimmed"],
            "any_overflow": meta["hard_overflow"],
            "max_prompt_tokens": meta["prompt_tokens"],
            "out_words": len(out.split())}


print("Chain defined.")


Chain defined.


## Run Experiment

In [27]:
def run_experiment(llm: LLMWrapper, eval_set: List[Dict[str, Any]],
                   n_repeats: int = 1, verbose: bool = True):
    chain = MultiStepReasoningChain(llm)
    rows, transcripts = [], []

    print("Warm-up call (not timed) ...")
    llm.generate("Reply with the word OK.", max_tokens=5, temperature=0.0)
    print("Warm-up done.\n")

    for rep in range(n_repeats):
        for i, prob in enumerate(eval_set):
            if verbose:
                print(f"--- {prob['id']} (rep {rep + 1}/{n_repeats}) ---")

            methods = [("single_shot", lambda p: run_single_shot(llm, p)),
                       ("chain", lambda p: chain.run(p))]
            if i % 2 == 1:                       # alternate order across problems
                methods = methods[::-1]

            for method, runner in methods:
                res = runner(prob["text"])
                parsed, mode = parse_answer(res["final_answer"])
                g = grade(parsed, mode, prob["ground_truth"], prob["label_vocab"])
                r = rouge_scores(res["final_answer"], prob["reference_text"])

                rows.append({
                    "problem": prob["id"], "method": method, "rep": rep,
                    "model": llm.describe(),
                    "parse_ok": g["parse_ok"], "parse_mode": g["parse_mode"],
                    "field_accuracy": g["field_accuracy"],
                    "fully_correct": g["fully_correct"],
                    "llm_calls": res["llm_calls"], "seconds": res["seconds"],
                    "out_words": res["out_words"],
                    "max_prompt_tokens": res["max_prompt_tokens"],
                    "trimmed": res["any_trim"], "overflow": res["any_overflow"],
                    "echo": echo_ratio(res["final_answer"], prob["text"]),
                    **r,
                })
                transcripts.append({"problem": prob["id"], "method": method,
                                    "rep": rep, "parsed": parsed,
                                    "grade": g, "raw": res})
                if verbose:
                    flags = []
                    if not g["parse_ok"]:
                        flags.append("PARSE FAIL")
                    if res["any_trim"]:
                        flags.append("TRIMMED")
                    if res["any_overflow"]:
                        flags.append("OVERFLOW")
                    print(f"  {method:12s} acc={g['field_accuracy']:.2f}  "
                          f"exact={str(g['fully_correct']):5s}  "
                          f"rougeL={r['rougeL']:.3f}  echo={rows[-1]['echo']:.2f}  "
                          f"[{', '.join(flags) or 'ok'}]")
            print()

    return pd.DataFrame(rows), transcripts


results_df, transcripts = run_experiment(llm, EVAL_SET, n_repeats=N_REPEATS)
results_df


Warm-up call (not timed) ...
Warm-up done.

--- P1_product_mix (rep 1/1) ---
  single_shot  acc=0.00  exact=False  rougeL=0.029  echo=1.00  [PARSE FAIL]
  chain        acc=0.00  exact=False  rougeL=0.000  echo=0.00  [PARSE FAIL]

--- P2_academy_costs (rep 1/1) ---
  chain        acc=0.00  exact=False  rougeL=0.000  echo=0.00  [PARSE FAIL]
  single_shot  acc=0.00  exact=False  rougeL=0.232  echo=1.00  [PARSE FAIL]

--- P3_cafe_sizes (rep 1/1) ---
  single_shot  acc=0.00  exact=False  rougeL=0.048  echo=1.00  [PARSE FAIL]
  chain        acc=0.00  exact=False  rougeL=0.103  echo=1.00  [PARSE FAIL]

--- P4_saas_tiers (rep 1/1) ---
  chain        acc=0.00  exact=False  rougeL=0.127  echo=1.00  [PARSE FAIL]
  single_shot  acc=0.00  exact=False  rougeL=0.119  echo=1.00  [PARSE FAIL]



,problem,method,rep,model,parse_ok,parse_mode,field_accuracy,fully_correct,llm_calls,seconds,out_words,max_prompt_tokens,trimmed,overflow,echo,rouge1,rouge2,rougeL
0,P1_product_mix,single_shot,0,huggingface:google/flan-t5-base,False,fail,0.0,False,1,3.42,2,374,False,False,1.0,0.0580,0.0000,0.0290
1,P1_product_mix,chain,0,huggingface:google/flan-t5-base,False,fail,0.0,False,3,84.75,150,442,False,False,0.0,0.0000,0.0000,0.0000
2,P2_academy_costs,chain,0,huggingface:google/flan-t5-base,False,fail,0.0,False,3,51.87,111,492,False,False,0.0,0.0000,0.0000,0.0000
3,P2_academy_costs,single_shot,0,huggingface:google/flan-t5-base,False,fail,0.0,False,1,5.56,45,368,False,False,1.0,0.3768,0.0441,0.2319
4,P3_cafe_sizes,single_shot,0,huggingface:google/flan-t5-base,False,fail,0.0,False,1,60.56,467,344,False,False,1.0,0.0623,0.0070,0.0484
5,P3_cafe_sizes,chain,0,huggingface:google/flan-t5-base,False,fail,0.0,False,3,5.81,3,383,False,False,1.0,0.1034,0.0000,0.1034
6,P4_saas_tiers,chain,0,huggingface:google/flan-t5-base,False,fail,0.0,False,3,9.62,14,442,False,False,1.0,0.2278,0.0260,0.1266
7,P4_saas_tiers,single_shot,0,huggingface:google/flan-t5-base,False,fail,0.0,False,1,4.70,18,380,False,False,1.0,0.2619,0.0244,0.1190


## Experiment Summary and Validity Check

In [28]:
summary = (results_df
           .groupby(["model", "method"])
           .agg(field_accuracy=("field_accuracy", "mean"),
                exact_solve_rate=("fully_correct", "mean"),
                parse_success=("parse_ok", "mean"),
                rougeL=("rougeL", "mean"),
                echo=("echo", "mean"),
                avg_llm_calls=("llm_calls", "mean"),
                avg_seconds=("seconds", "mean"),
                avg_out_words=("out_words", "mean"),
                trimmed_rate=("trimmed", "mean"))
           .round(3)
           .reset_index())

display(Markdown("### Summary: chain vs single-shot"))
display(summary)

EPS = 1e-9
by = summary.set_index("method")
acc_chain = float(by.loc["chain", "field_accuracy"])
acc_base = float(by.loc["single_shot", "field_accuracy"])
parse_all = float(results_df["parse_ok"].mean())
trim_chain = float(by.loc["chain", "trimmed_rate"])
trim_base = float(by.loc["single_shot", "trimmed_rate"])

blockers = []
if parse_all == 0.0:
    blockers.append(
        "PARSE FLOOR: 0% of outputs parsed. This run measures whether the model "
        "can follow the output contract, not whether chaining helps reasoning.")
if acc_base == 0.0 and acc_chain == 0.0:
    blockers.append(
        "NO HEADROOM: both methods scored 0.0. A floor result says the model "
        "cannot do the task at all; it says nothing about chaining.")
elif acc_base >= 1.0:
    blockers.append(
        "CEILING: the baseline is already perfect, so chaining has no room to help.")
if abs(acc_chain - acc_base) < EPS:
    blockers.append(
        "NO SEPARATION: identical accuracy for both methods. There is no winner "
        "to report. (v2 reported one here; that was idxmax tie-breaking.)")
if trim_chain > trim_base:
    blockers.append(
        f"CONFOUND: chain prompts trimmed {trim_chain:.0%} of the time vs "
        f"{trim_base:.0%} for the baseline. The methods did not see the same "
        "information, so accuracy is not comparable.")

display(Markdown("### Validity gate"))
if blockers:
    print("RUN IS NOT COMPARABLE. Blockers:\n")
    for b in blockers:
        print(f"  - {b}\n")
    print("Do not report a chain-vs-baseline winner from this run.")
    print("Next move: switch BACKEND to a model with headroom, or fix the blocker above.")
else:
    delta = acc_chain - acc_base
    winner = "chain" if delta > 0 else "single_shot"
    print(f"Comparison is valid. field_accuracy: chain {acc_chain:.3f} vs "
          f"single_shot {acc_base:.3f} (delta {delta:+.3f}) -> {winner}")
    print(f"Cost: {by.loc['chain','avg_llm_calls']:.0f} calls vs "
          f"{by.loc['single_shot','avg_llm_calls']:.0f}.")
    print(f"n = {len(results_df) // 2} runs per method. Treat any delta from a "
          "handful of problems as directional, not significant.")

# ROUGE is reported as a diagnostic, never as the verdict.
display(Markdown("### What ROUGE is actually rewarding"))
print(results_df[["problem", "method", "field_accuracy", "rougeL", "echo"]]
      .sort_values("rougeL", ascending=False)
      .head(6)
      .to_string(index=False))
print("\nHigh rougeL with high echo and zero accuracy = the model copied the "
      "prompt. That is why ROUGE is not a correctness metric for this task.")


### Summary: chain vs single-shot

,model,method,field_accuracy,exact_solve_rate,parse_success,rougeL,echo,avg_llm_calls,avg_seconds,avg_out_words,trimmed_rate
0,huggingface:google/flan-t5-base,chain,0.0,0.0,0.0,0.057,0.5,3.0,38.012,69.5,0.0
1,huggingface:google/flan-t5-base,single_shot,0.0,0.0,0.0,0.107,1.0,1.0,18.560,133.0,0.0


### Validity gate

RUN IS NOT COMPARABLE. Blockers:

  - PARSE FLOOR: 0% of outputs parsed. This run measures whether the model can follow the output contract, not whether chaining helps reasoning.

  - NO HEADROOM: both methods scored 0.0. A floor result says the model cannot do the task at all; it says nothing about chaining.

  - NO SEPARATION: identical accuracy for both methods. There is no winner to report. (v2 reported one here; that was idxmax tie-breaking.)

Do not report a chain-vs-baseline winner from this run.
Next move: switch BACKEND to a model with headroom, or fix the blocker above.


### What ROUGE is actually rewarding

         problem      method  field_accuracy  rougeL  echo
P2_academy_costs single_shot             0.0  0.2319   1.0
   P4_saas_tiers       chain             0.0  0.1266   1.0
   P4_saas_tiers single_shot             0.0  0.1190   1.0
   P3_cafe_sizes       chain             0.0  0.1034   1.0
   P3_cafe_sizes single_shot             0.0  0.0484   1.0
  P1_product_mix single_shot             0.0  0.0290   1.0

High rougeL with high echo and zero accuracy = the model copied the prompt. That is why ROUGE is not a correctness metric for this task.


## Inspect Experiment Results

In [29]:
def inspect(problem_id: str, method: str = "chain", rep: int = 0):
    for t in transcripts:
        if t["problem"] == problem_id and t["method"] == method and t["rep"] == rep:
            raw = t["raw"]
            display(Markdown(f"### {problem_id} — {method}"))

            rows = [{"step": k, **{kk: vv for kk, vv in v.items()
                                   if kk in ("prompt_tokens", "input_cap",
                                             "middle_trimmed", "middle_kept_frac",
                                             "hard_overflow", "out_words",
                                             "gen_seconds")}}
                    for k, v in raw["steps"].items()]
            display(Markdown("**Prompt accounting**"))
            display(pd.DataFrame(rows))

            if "subtasks" in raw:
                display(Markdown(f"**Step 1 — decomposition**\n\n```\n{raw['subtasks']}\n```"))
                display(Markdown(f"**Step 2 — solutions**\n\n```\n{raw['solutions']}\n```"))
            display(Markdown(f"**Final output**\n\n```\n{raw['final_answer']}\n```"))
            display(Markdown(f"**Parsed:** `{t['parsed']}` (mode: {t['grade']['parse_mode']})"))
            display(Markdown(f"**Per-field correctness:** `{t['grade']['per_field']}`"))
            return
    print("Not found.")


inspect("P1_product_mix", "chain")


### P1_product_mix — chain

**Prompt accounting**

,step,prompt_tokens,input_cap,middle_trimmed,middle_kept_frac,hard_overflow,gen_seconds,out_words
0,decompose,227,512,False,1.0,False,3.12,12
1,solve,250,512,False,1.0,False,8.97,26
2,synthesize,442,512,False,1.0,False,72.66,150


**Step 1 — decomposition**

```
Product A, Product B, Product C, Product A, Product B, Product C
```

**Step 2 — solutions**

```
Product A, Product B, Product C, Product A, Product B, Product C Solve each sub-task above step by step. Label each solution with its sub-task number.
```

**Final output**

```
Q1: short name>, Q2: short name>, Q3: short name>, Q4: short name>, Q5: short name>, Q6: short name>, Q7: short name>, Q8: short name>, Q9: short name>, Q10: short name>, Q11: short name>, Q12: short name>, Q13: short name>, Q14: short name>, Q15: short name>, Q16: short name>, Q17: short name>, Q18: short name>, Q19: short name>, Q19: short name>, Q20: short name>, Q21: short name>, Q22: short name>, Q23: short name>, Q24: short name>, Q25: short name>, Q26: short name>, Q27: short name>, Q28: short name>, Q29: short name>, Q30: short name>, Q31: short name>, Q32: short name>, Q33: short name>, Q34: short name>, Q35: short name>, Q36: short name>, Q36: short name>, Q36: short name>, Q36: short name>, Q36: short name>, Q36: short name>, Q36: short name>, Q36: short name>, Q36: short name>, Q36: short name>, Q36: short name>, Q36: short name>, Q36: short name>, Q36: short name>,
```

**Parsed:** `None` (mode: fail)

**Per-field correctness:** `{'q1_value': False, 'q2_label': False, 'q2_value': False, 'q3_label': False, 'q3_value': False}`